# 양방향 LSTM

<img src="image/nlp_image/bidirectional_lstm.jpg">

- RNN이나 LSTM은 시퀀스 또는 시계열 데이터 처리에 특화되어 은닉층에서 과거의 정보를 기억할 수 있음

- 그러나 순환 구조의 특성상 데이터가 입력 순으로 처리되기 때문에 이전 시점의 정보만 활용할 수 밖에 없다는 단점이 존재
    - 문장이 길어질수록 성능이 저하
    - 예) ios앱 (개발)은 맥북이 필요합니다
        - 위의 경우에 ios와 앱 이라는 단어만으로 개발 이라는 단어를 유추하기 힘듦
        - 문장의 앞부분보다 뒷부분에 중요한 정보가 있음

- 양방향 LSTM(Bidirectional LSTM)
    - 기존 LSTM 계층에 역방향으로 처리하는 LSTM 계층을 추가해 양방향에서 문장의 패턴을 분석할 수 있도록 구성
    - 입력 문장을 양방향에서 처리하기 때문에 시퀀스 길이가 길어져도 정보 손실 없이 처리가 가능

# 개체명 인식(Named Entity Recognition)

- 각 개체의 유형을 인식

- 문장 내에 포함된 어떤 단어가 인물, 장소, 날짜 등을 의미하는 단어인지 인식하는 것

- 예) 날짜와 지역에 대해 개체 인식을 할 수 있는 모델이 있다고 가정할 경우
    - 입력 문장 : 내일 부산 날씨 알려줘
    - 문장 의도 : 날씨 요청
    - 개체명 인식 결과 : 내일 - 날짜, 부산 - 지역

- 단순한 질문 형태라면 개체명 사전을 구축해 해당 단어들과 매핑되는 개체명을 찾을 수도 있음
    - 문장 구조가 복잡하거나 문맥에 따라 단어의 의미가 바뀐다면 딥러닝 모델을 활용해야 함
 
- 개체명 사전 구축 방식은 신조어나 사전에 포함되지 않은 단어는 처리 불가능하며 사람이 직접 사전 데이터를 관리해야 하기 때문에 관리비용이 많이 필요함

## BIO 표기법

- Beginning, Inside, Outside 의 약자

- 각 토큰마다 태그를 붙이기 위해 사용

- Beginning : 개체명이 시작되는 단어에 "B-개체명"으로 태그

- Inside : "B-개체명" 과 연결되는 단어일 때 "I-개체명"으로 태그

- Outside : 개체명 이외의 모든 토큰에 태그
    - 예) 오늘부터 샤닐 길동은 삼성전자에 근무합니다
        - 오늘 -> B-date
        - 부터 -> O
        - 샤닐 -> B-Person
        - 길동 -> I-Person
        - 은 -> O
        - 삼성 -> B-Company
        - 전자 -> I-Company
        - 에 -> O
        - 근무 -> O
        - 합니다 -> O
 
- 두 개 이상의 토큰이 하나의 개체를 구성하는 경우가 많기 때문에 BIO 표기법을 사용

## 국립국어원 언어정보나눔터 개체명 인식 모델을 위한 말뭉치

- ; 으로 시작하는 문장 : 원본 문장

- $ 로 시작하는 문장 : 해당 문장에서 NER 처리된 결과

- 개체명 인식 모델은 단어 토큰을 입력했을 때 출력되는 NER 태그값을 예측
    - 예) 삼성전자 를 입력한다면 B_OG(단체) 태그가 출력되도록 학습

In [1]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import preprocessing
from tensorflow import keras
import numpy as np
from sklearn.model_selection import train_test_split
# F1 스코어 계산을 위해 사용
from seqeval.metrics import f1_score, classification_report

In [2]:
# 학습 파일 불러오기
def read_file(file_name):
    sents = []

    with open(file_name, "r", encoding = "utf-8") as f:
        lines = f.readlines()
        
        for idx, l in enumerate(lines):
            if l[0] == ";" and lines[idx + 1][0] == "$":
                this_sent = []

            elif l[0] == "$" and lines[idx - 1][0] == ";":
                continue

            elif l[0] == "\n":
                sents.append(this_sent)

            else:
                this_sent.append(tuple(l.split()))
    return sents

In [3]:
# 학습용 말뭉치 데이터를 불러옴
corpus = read_file("./data/train.txt")

In [4]:
corpus[0]

[('1', '한편', 'NNG', 'O'),
 ('1', ',', 'SP', 'O'),
 ('2', 'AFC', 'SL', 'O'),
 ('2', '챔피언스', 'NNG', 'O'),
 ('2', '리그', 'NNG', 'O'),
 ('3', 'E', 'SL', 'B_OG'),
 ('3', '조', 'NNG', 'I'),
 ('3', '에', 'JKB', 'O'),
 ('4', '속하', 'VV', 'O'),
 ('4', 'ㄴ', 'ETM', 'O'),
 ('5', '포항', 'NNP', 'O'),
 ('6', '역시', 'MAJ', 'O'),
 ('7', '대회', 'NNG', 'O'),
 ('8', '8강', 'NNG', 'O'),
 ('9', '진출', 'NNG', 'O'),
 ('9', '이', 'JKS', 'O'),
 ('10', '불투명', 'NNG', 'O'),
 ('10', '하', 'VV', 'O'),
 ('10', '다', 'EC', 'O'),
 ('11', '.', 'SF', 'O')]

In [6]:
# 말뭉치 데이터에서 단어와 BIO 태그만 불러와 학습용 데이터셋 생성
sentences = []
tags = []

for t in corpus:
    tagged_sentence = []
    sentence = []
    bio_tag = []

    for w in t:
        tagged_sentence.append((w[1], w[3]))
        sentence.append(w[1])
        bio_tag.append(w[3])

    sentences.append(sentence)
    tags.append(bio_tag)

- 단어와 BIO 태그만 이용해 학습용 데이터셋을 생성

In [9]:
print("샘플 크기 :", len(sentences))
print("0번째 샘플 문장 시퀀스 :")
print(sentences[0])
print("0번째 샘플 bio 태그 :")
print(tags[0])
print("샘플 문장 시퀀스 최대 길이 :", max(len(l) for l in sentences))
print("샘플 문장 시퀀스 평균 길이 :", (sum(map(len, sentences)) / len(sentences)))

샘플 크기 : 3555
0번째 샘플 문장 시퀀스 :
['한편', ',', 'AFC', '챔피언스', '리그', 'E', '조', '에', '속하', 'ㄴ', '포항', '역시', '대회', '8강', '진출', '이', '불투명', '하', '다', '.']
0번째 샘플 bio 태그 :
['O', 'O', 'O', 'O', 'O', 'B_OG', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
샘플 문장 시퀀스 최대 길이 : 168
샘플 문장 시퀀스 평균 길이 : 34.03909985935302


In [10]:
# 토크나이저 정의
sent_tokenizer = preprocessing.text.Tokenizer(oov_token = "OOV")
sent_tokenizer.fit_on_texts(sentences)

tag_tokenizer = preprocessing.text.Tokenizer(lower = False)
tag_tokenizer.fit_on_texts(tags)

In [11]:
# 단어 사전 및 태그 사전 크기
vocab_size = len(sent_tokenizer.word_index) + 1
tag_size = len(tag_tokenizer.word_index) + 1

print("BIO 태그 사전 크기 :", tag_size)
print("단어 사전 크기 :", vocab_size)

BIO 태그 사전 크기 : 8
단어 사전 크기 : 13834


In [12]:
tag_tokenizer.word_index

{'O': 1, 'I': 2, 'B_OG': 3, 'B_PS': 4, 'B_DT': 5, 'B_LC': 6, 'B_TI': 7}

In [13]:
# 학습용 단어 시퀀스 생성
x_train = sent_tokenizer.texts_to_sequences(sentences)
y_train = tag_tokenizer.texts_to_sequences(tags)

print(x_train[0])
print(y_train[0])

[183, 11, 4276, 884, 162, 931, 402, 10, 2608, 7, 1516, 608, 145, 1361, 414, 4, 6347, 2, 8, 3]
[1, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [14]:
# index to word / index to NER 정의
index_to_word = sent_tokenizer.index_word # 시퀀스 인덱스를 단어로 변환하기 위해 사용
index_to_ner = tag_tokenizer.index_word # 시퀀스 인덱스를 NER로 변환하기 위해 사용
index_to_ner[0] = "PAD"

In [15]:
index_to_ner

{1: 'O',
 2: 'I',
 3: 'B_OG',
 4: 'B_PS',
 5: 'B_DT',
 6: 'B_LC',
 7: 'B_TI',
 0: 'PAD'}

In [17]:
# 시퀀스 패딩 처리
max_len = 40
x_train = preprocessing.sequence.pad_sequences(x_train, padding = "post", maxlen = max_len)
y_train = preprocessing.sequence.pad_sequences(y_train, padding = "post", maxlen = max_len)

- 벡터의 크기는 시퀀스의 평균 길이보다 넉넉하게 40으로 정의

In [18]:
# 학습 데이터와 테스트 데이터를 8 : 2  비율로 분리
x_train, x_test, y_train, y_test = train_test_split(x_train, y_train, test_size = 0.2,
                                                    random_state = 31)

In [19]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((2844, 40), (711, 40), (2844, 40), (711, 40))

In [20]:
y_train[0]

array([3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], dtype=int32)

In [21]:
# 출력 데이터를 원-핫 인코딩
y_train = tf.keras.utils.to_categorical(y_train, num_classes = tag_size)
y_test= tf.keras.utils.to_categorical(y_test, num_classes = tag_size)

In [22]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((2844, 40), (711, 40), (2844, 40, 8), (711, 40, 8))

In [23]:
x_train[0]

array([  79,  104,  434, 1920,   10,  341,   25,    7,  199, 1938,  435,
        942,    6,  105, 1078,    5,  421,   39,   76,   14,  114, 4363,
         13,   43, 1394,   19,  993, 1001,   12,   80,   15, 1088, 3336,
          5,    2,   51,   18,    8,    3,    0], dtype=int32)

In [24]:
y_train[0]

array([[0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 

In [27]:
# 모델 정의(Bi-LSTM)
model = keras.Sequential()
model.add(keras.Input(shape = (max_len,)))
model.add(keras.layers.Embedding(vocab_size, 64, mask_zero = True))
model.add(keras.layers.Bidirectional(keras.layers.LSTM(256, return_sequences = True,
                                                       dropout = 0.5, recurrent_dropout = 0.25)))
model.add(keras.layers.Dense(tag_size, activation = "softmax"))

- mask_zero = True : 0으로 패딩된 값을 마스킹하여 네트워크의 뒤로 전달되지 않게 만듦

<img src="image/nlp_image/time_distributed.jpg">

- TimeDistributed
    - 주로 시퀀스 데이터나 시계열 데이터를 다루는 신경망 모델에서 사용
    - 각 타임스텝에 동일한 레이어를 적용할 때 사용
        - 주로 RNN, CNN 과 결합
     
    - 입력 데이터의 각 타임스텝에 동일한 레이어(Dense, Conv 등)을 독립적으로 적용할 수 있게 해주는 래퍼(wrapper)
        - 시퀀스 데이터의 각 요소에 동일한 처리를 반복적으로 수행할 수 있음
     
    - 특징
        - 타임스텝별 독립 처리 : 각 타임스텝에 동일한 레이어를 독립적으로 적용
        - 레이어 재사용 : 레이어의 가중치가 모든 타임스텝에 공유되어 효율적인 학습이 가능
        - 유연성 : 다양한 레이어를 TimeDistributed 로 감싸서 시퀀스 데이터에 적용할 수 있음

In [28]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 40, 64)              │         885,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 40, 512)             │         657,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 40, 8)               │           4,104 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,546,888 (5.90 MB)

 Trainable params: 1,546,888 (5.90 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.compile(loss = "categorical_crossentropy", optimizer = "Adam", metrics = ["accuracy"])

In [30]:
es_cb = keras.callbacks.EarlyStopping(patience = 4, restore_best_weights = True)

In [31]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.2,
                                                  random_state = 31)

In [32]:
model.fit(x_train, y_train, batch_size = 64, epochs = 100, validation_data = (x_val, y_val),
          callbacks = [es_cb])

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 19s 347ms/step - accuracy: 0.6105 - loss: 0.9109 - val_accuracy: 0.6444 - val_loss: 0.5727
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 294ms/step - accuracy: 0.6164 - loss: 0.5493 - val_accuracy: 0.6445 - val_loss: 0.4945
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 259ms/step - accuracy: 0.6212 - loss: 0.4506 - val_accuracy: 0.6569 - val_loss: 0.3986
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 262ms/step - accuracy: 0.6329 - loss: 0.3539 - val_accuracy: 0.6634 - val_loss: 0.3450
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 254ms/step - accuracy: 0.6423 - loss: 0.2898 - val_accuracy: 0.6685 - val_loss: 0.3208
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 263ms/step - accuracy: 0.6500 - loss: 0.2542 - val_accuracy: 0.6687 - val_loss: 0.3113
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 275ms/step - accuracy: 0.6546 - loss: 0.2315 - val_accuracy: 0.6742 - val_loss: 0.3066
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 303ms/step - accuracy: 0.6605 - loss: 0.2085 - val_

In [33]:
print("평가 결과 : ", model.evaluate(x_test, y_test))

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.6370 - loss: 0.3224
평가 결과 :  [0.32244423031806946, 0.6370253562927246]


- BIO 태그는 실제 의미 있는 태그보다 의미 없는 O 태그가 대부분을 차지하고 있기 때문에 실제 성능과 무관하게 높은 점수가 나올 수 있음

- 따라서 개체명 인식에서는 F1 스코어가 주로 사용됨

- F1 스코어 : 정밀도와 재현율의 조화 평균

In [39]:
# 시퀀스를 NER 태그로 변환
def sequences_to_tag(sequences):
    result = []

    for sequence in sequences:
        temp = []
        for pred in sequence:
            pred_index = np.argmax(pred)
            temp.append(index_to_ner[pred_index].replace("PAD", "O"))
        result.append(temp)
    return result

In [40]:
x_test.shape

(711, 40)

In [36]:
# 테스트 데이터셋의 NER 예측
y_predicted = model.predict(x_test) # (711, 40) => (711, 40, 8)

23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step  


In [41]:
y_predicted.shape

(711, 40, 8)

In [42]:
pred_tags = sequences_to_tag(y_predicted) # 예측된 NER
test_tags = sequences_to_tag(y_test) # 실제 NER

In [43]:
pred_tags[0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B_DT',
 'I',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I',
 'I',
 'I',
 'I',
 'I',
 'I',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [44]:
test_tags[0]

['I',
 'I',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B_DT',
 'I',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B_DT',
 'I',
 'I',
 'I',
 'I',
 'B_DT',
 'I',
 'I',
 'I',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [45]:
print(classification_report(test_tags, pred_tags))
print(f"F1-score: {f1_score(test_tags, pred_tags)}")

C:\ProgramData\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: B_DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\ProgramData\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: B_OG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\ProgramData\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: B_TI seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\ProgramData\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: B_PS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\ProgramData\anaconda3\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: B_LC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


              precision    recall  f1-score   support

           _       0.41      0.42      0.42       591
         _DT       0.68      0.78      0.73       294
         _LC       0.59      0.15      0.24       343
         _OG       0.39      0.40      0.39       511
         _PS       0.71      0.09      0.16       389
         _TI       0.00      0.00      0.00        48

   micro avg       0.48      0.35      0.41      2176
   macro avg       0.46      0.31      0.32      2176
weighted avg       0.51      0.35      0.37      2176

F1-score: 0.40657301881791674


In [46]:
# 새로운 유형의 문장 NER 예측
word_to_index = sent_tokenizer.word_index
new_sentences = "삼성전자 출시 스마트폰 오늘 애플 도전장 내밀다".split()
new_x = []

for w in new_sentences:
    try:
        new_x.append(word_to_index.get(w, 1))
    except KeyError:
        # 모르는 단어의 경우 OOV
        new_x.append(word_to_index["OOV"])

In [47]:
print("새로운 유형의 시퀀스 :", new_x)
new_padded_seqs = preprocessing.sequence.pad_sequences([new_x], padding = "post", maxlen = max_len)

새로운 유형의 시퀀스 : [531, 307, 1476, 286, 1507, 6766, 1]


In [48]:
# NER 예측
p = model.predict(np.array([new_padded_seqs[0]]))
p = np.argmax(p, axis = -1) # 예측된 NER 인덱스값 추출

print("{:10} {:5}".format("단어", "예측된 NER"))
print("-" * 50)

for w, pred in zip(new_sentences, p[0]):
    print("{:10} {:5}".format(w, index_to_ner[pred]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
단어         예측된 NER
--------------------------------------------------
삼성전자       B_OG 
출시         O    
스마트폰       O    
오늘         B_DT 
애플         I    
도전장        I    
내밀다        I    
